# LG Aimers HGB Feature Selection V2

Colab 전용 실행 노트북입니다.

순서: Drive 연결 → GitHub V2 브랜치 clone → 데이터 경로 확인 → quick 실행 → 결과 확인 → full 실행.


In [ ]:
from pathlib import Path

# 이미 /content/drive/MyDrive 가 정상 마운트되어 있으면 그대로 사용합니다.
# 그렇지 않으면 기존 /content/drive 충돌을 피하기 위해 전용 mountpoint를 사용합니다.
if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    print("Existing Drive mount detected:", DRIVE_ROOT)
else:
    from google.colab import drive
    MOUNT_POINT = "/content/gdrive_hgb_v2"
    drive.mount(MOUNT_POINT)
    DRIVE_ROOT = Path(MOUNT_POINT) / "MyDrive"
    print("Drive mounted at:", DRIVE_ROOT)

assert DRIVE_ROOT.is_dir(), f"MyDrive not available: {DRIVE_ROOT}"


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/lg_aimers_experiment_lab_v2")
BRANCH = "agent/hgb-feature-selection-v2"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", BRANCH,
    "https://github.com/tswaincae1221/lg_aimers_experiment_lab.git",
    str(REPO_DIR),
], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
print("Repository ready:", REPO_DIR)


In [ ]:
def resolve_drive_file(filename: str, preferred_folder: str = "aimers_data") -> Path:
    preferred = DRIVE_ROOT / preferred_folder / filename
    if preferred.is_file():
        return preferred
    matches = sorted(path for path in DRIVE_ROOT.rglob(filename) if path.is_file())
    if len(matches) == 1:
        print(f"Auto-resolved {filename}: {matches[0]}")
        return matches[0]
    if not matches:
        raise FileNotFoundError(f"{filename} not found under {DRIVE_ROOT}")
    print(f"Multiple {filename} candidates found:")
    for path in matches[:30]:
        print(" -", path)
    raise RuntimeError(f"Set the intended {filename} path manually.")

TRAIN_PATH = resolve_drive_file("train.csv")
TRACKMAN_PATH = resolve_drive_file("trackman_history.csv")
MAPPING_PATH = REPO_DIR / "resources" / "pitcher_trackman_mapping.csv"

print("TRAIN   =", TRAIN_PATH)
print("TRACKMAN=", TRACKMAN_PATH)
print("MAPPING =", MAPPING_PATH)
assert TRAIN_PATH.is_file() and TRACKMAN_PATH.is_file() and MAPPING_PATH.is_file()


In [ ]:
# 처음에는 quick으로 실행하세요. quick 성공 후 "full"로 바꿔 다시 실행합니다.
MODE = "quick"  # "quick" or "full"
OUTPUT_DIR = DRIVE_ROOT / "aimers_data" / "results" / f"hgb_feature_selection_v2_{MODE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("MODE      =", MODE)
print("OUTPUT_DIR=", OUTPUT_DIR)


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "-m", "src.hgb_feature_selection_v2",
    "--train", str(TRAIN_PATH),
    "--trackman", str(TRACKMAN_PATH),
    "--mapping", str(MAPPING_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--validation-season", "2024",
    "--tuning-season", "2023",
    "--mode", MODE,
]
print("Running:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
import json
import pandas as pd

print("=== model_scores.csv ===")
display(pd.read_csv(OUTPUT_DIR / "model_scores.csv"))

print("=== block_ablation.csv ===")
display(pd.read_csv(OUTPUT_DIR / "block_ablation.csv"))

print("=== permutation_importance.csv (top 30) ===")
display(pd.read_csv(OUTPUT_DIR / "permutation_importance.csv").head(30))

print("=== lofo_results.csv ===")
display(pd.read_csv(OUTPUT_DIR / "lofo_results.csv"))

print("=== run_summary.json ===")
print(json.dumps(json.load(open(OUTPUT_DIR / "run_summary.json", encoding="utf-8")), ensure_ascii=False, indent=2))


## Full 실행

Quick 결과가 정상이라면 `MODE = "full"` 로 바꾸고 **MODE 셀 → pipeline 실행 셀 → 결과 확인 셀** 순서로 다시 실행하세요.

Full 모드는 2019–2023 전체 학습 / 2024 전체 검증이며, Trackman도 전체 이력을 사용합니다.
